# 🚀 OpenSource Clipping — Quick Start Notebook

> **⚡ Quick Start**: If you just want to get up and running as fast as possible,
> simply run all cells from top to bottom. The only thing you need to change is
> the **YouTube URL** and **Gemini API key**, or select **vLLM** in the provider section below.

---

## What is this?

This notebook is a **minimal, beginner-friendly** implementation of
[OpenSource Clipping](https://github.com/NaufalRizqullah/opensource-clipping) —
an AI-powered tool that automatically transforms long-form videos into
cinematic short-form highlight clips with:

- 🎯 AI-curated highlight moments (Google Gemini or your vLLM server)
- 🎙️ Word-level transcription (powered by Faster-Whisper)
- 📱 Smart face-tracking auto-framing for vertical (9:16) format
- 💬 Karaoke-style animated subtitles
- 🎬 Cinematic hook teaser intro with glitch transition

## How to use this notebook

| Step | Section | What to do |
|------|---------|------------|
| 1 | **Setup** | Run the setup cells to clone the repo & install dependencies |
| 2 | **Configuration** | Choose Gemini/vLLM, configure credentials, and set your clip preferences |
| 3 | **Run** | Execute the pipeline — sit back and wait for your clips! |
| 4 | **Download** | Download the generated clips from the `outputs/` folder |

### ⚠️ Requirements

- **Runtime**: Set your Colab runtime to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`)
- **AI provider**: A [Google Gemini API key](https://aistudio.google.com/apikey) for Gemini, or a running vLLM server (optional local setup included)

---

## 1. Setup — Clone Repository & Install Dependencies

This section clones the OpenSource Clipping repository and installs all
required Python packages. It also installs **FFmpeg** (needed for video
processing) and **Deno** (used internally by some modules).

> 💡 **Tip**: This cell only needs to run once per Colab session.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 1a. Clone the repository into the current working directory
# ──────────────────────────────────────────────────────────────────────
!rm -rf ./* ./.*
!git clone https://github.com/NaufalRizqullah/opensource-clipping.git .

## Choose the AI provider

Set these options **before running the dependency installation cells**:

- `AI_PROVIDER = "gemini"` keeps the default flow and requires `GOOGLE_API_KEY`.
- `AI_PROVIDER = "vllm"` uses your vLLM server without a Google API key.
- **Existing server:** keep `START_LOCAL_VLLM = False`, set `VLLM_BASE_URL` and the exact served `VLLM_MODEL`, and add `VLLM_API_KEY` to Secrets only if authentication is required.
- **Host in this notebook:** set `START_LOCAL_VLLM = True`. Later cells prepare separate Python 3.12 application/server environments, start vLLM, and evaluate a short transcript. Gemma 4 Q4 is the same experimental preset as the [dedicated notebook](kaggle-vllm-gemma4.ipynb).

Local hosting requires Linux x86_64 and a supported GPU (T4 or SM 7.5+; P100 is unsupported). Memory fit still requires validation. Whisper uses CPU/int8 while local vLLM occupies the GPU; start with a small Whisper model and one clip. An external server leaves your Whisper settings unchanged.

`VLLM_CONTEXT_LIMIT` is the total context ceiling; `VLLM_MAX_TOKENS` must be smaller to leave room for the prompt. `VLLM_TIMEOUT` is the analysis/retry budget in seconds. An existing endpoint must expose both `/v1` and `/tokenize` and be reachable from this notebook; `127.0.0.1` refers to this runtime. See the [provider guide](../docs/providers.md) and [Kaggle setup guide](../docs/KAGGLE_VLLM.md).

Keep the server running while clipping. After changing provider/server settings, rerun the following setup cells; stop an existing local server before reinstalling or changing its model/key.


In [ ]:
# AI provider configuration — edit before installing dependencies
import os
from pathlib import Path
import subprocess
import sys

AI_PROVIDER = "gemini"  # "gemini" (default) or "vllm"
START_LOCAL_VLLM = False  # True: install/start the optional server in this notebook
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://127.0.0.1:8001")
VLLM_MODEL = os.environ.get("VLLM_MODEL", "google/gemma-4-E2B-it-qat-w4a16-ct")
VLLM_CONTEXT_LIMIT = os.environ.get("VLLM_CONTEXT_LIMIT", "16384")
VLLM_MAX_TOKENS = os.environ.get("VLLM_MAX_TOKENS", "4096")
VLLM_TIMEOUT = os.environ.get("VLLM_TIMEOUT", "900")
GPU_INDEX = 0
VLLM_PORT = 8001

if AI_PROVIDER not in {"gemini", "vllm"}:
    raise ValueError('Choose AI_PROVIDER = "gemini" or "vllm".')
if START_LOCAL_VLLM and AI_PROVIDER != "vllm":
    raise ValueError('START_LOCAL_VLLM requires AI_PROVIDER = "vllm".')

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
APP_PYTHON = sys.executable
APP_ENV = None
cache_env = {}
LOCAL_VLLM_PREPARED = False
LOCAL_VLLM_READY = False
print(f"AI provider: {AI_PROVIDER}; local vLLM setup: {START_LOCAL_VLLM}")


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# 1b. Install system dependencies (FFmpeg + Deno) and Python packages
# ──────────────────────────────────────────────────────────────────────
import os

# FFmpeg — required for video encoding/decoding
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Deno — used by internal subtitle rendering modules
!curl -fsSL https://deno.land/install.sh | sh
os.environ["PATH"] += ":/root/.deno/bin"

# Python dependencies
if not START_LOCAL_VLLM:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
else:
    print("Application dependencies will be installed in the isolated vLLM setup below.")

## 2. Configuration — Set Your Preferences

Configure your clip preferences here. If YouTube requires cookies, also set `YOUTUBE_COOKIES_FILE` in the cookie setup cell before running the pipeline.

| Parameter | Description | Default |
|-----------|-------------|---------|
| `YOUTUBE_URL` | The YouTube video URL you want to clip | *(required)* |
| `TOTAL_CLIPS` | How many highlight clips to generate | `3` |
| `WHISPER_MODEL` | Whisper model size for transcription accuracy | `large-v3` |
| `WHISPER_DEVICE` | Device for Whisper inference (`cuda` or `cpu`) | `cuda` |
| `WHISPER_COMPUTE_TYPE` | Compute precision (`float16` for Colab T4, `float32` for CPU) | `float16` |

### Whisper Model Options

| Model | Speed | Accuracy | VRAM |
|-------|-------|----------|------|
| `tiny` | ⚡⚡⚡⚡⚡ | ★☆☆☆☆ | ~1 GB |
| `base` | ⚡⚡⚡⚡ | ★★☆☆☆ | ~1 GB |
| `small` | ⚡⚡⚡ | ★★★☆☆ | ~2 GB |
| `medium` | ⚡⚡ | ★★★★☆ | ~5 GB |
| `large-v3` | ⚡ | ★★★★★ | ~10 GB |

> 💡 **Recommendation**: Use `large-v3` with `float16` on Colab T4 GPU for best
> results. If you're on a free Kaggle notebook or CPU-only environment, use
> `small` or `medium` with `float32`.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Edit the values below to match your preferences
# ══════════════════════════════════════════════════════════════════════

# ── YouTube URL ──────────────────────────────────────────────────────
# Paste the YouTube video URL you want to generate clips from.
YOUTUBE_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # <-- CHANGE THIS

# ── Total Clips ─────────────────────────────────────────────────────
# Number of highlight clips to generate from the video.
# More clips = longer processing time. Start small (1-3) for testing.
TOTAL_CLIPS = 3

# ── Whisper Configuration ────────────────────────────────────────────
# These control the speech-to-text transcription step.
WHISPER_MODEL = "large-v3"        # Model size (see table above)
WHISPER_DEVICE = "cuda"            # "cuda" for GPU, "cpu" for CPU-only
WHISPER_COMPUTE_TYPE = "float16"   # "float16" for Colab T4, "float32" for CPU/Kaggle

# ══════════════════════════════════════════════════════════════════════
#  OPTIONAL — Advanced settings (safe to leave as defaults)
# ══════════════════════════════════════════════════════════════════════

ASPECT_RATIO = "9:16"              # Output aspect ratio: "9:16", "16:9", "1:1"
FONT_STYLE = "HORMOZI"             # Subtitle style: "DEFAULT", "HORMOZI", "CINEMATIC", "STORYTELLER"
HOOK_DURATION = 3                  # Hook teaser duration in seconds
WORDS_PER_SUBTITLE = 5             # Max words per subtitle group

print("✅ Configuration loaded!")
print(f"   URL           : {YOUTUBE_URL}")
print(f"   Total Clips   : {TOTAL_CLIPS}")
print(f"   Whisper Model : {WHISPER_MODEL} ({WHISPER_DEVICE}, {WHISPER_COMPUTE_TYPE})")
print(f"   Aspect Ratio  : {ASPECT_RATIO}")
print(f"   Font Style    : {FONT_STYLE}")

## 3. Provider credentials

For **Gemini**, add `GOOGLE_API_KEY` to Colab Secrets (key icon, enable notebook access) or Kaggle **Add-ons → Secrets**. Get it from [Google AI Studio](https://aistudio.google.com/apikey).

For **vLLM**, Google API keys are not required. Add `VLLM_API_KEY` only when the server requires it. Local setup generates an internal key in memory if none is configured; keep it unchanged while the server runs.

Optional: `PEXELS_API_KEY` enables B-roll; `HF_TOKEN` enables authenticated Hugging Face downloads and podcast diarization. Environment variables take precedence over Secrets. Missing optional secrets do not stop the notebook.


In [ ]:
# Load provider credentials from the environment, Colab Secrets, or Kaggle Secrets
secret_readers = []
try:
    from kaggle_secrets import UserSecretsClient
    secret_readers.append(UserSecretsClient().get_secret)
except ImportError:
    pass
except Exception as exc:
    print(f"Kaggle Secrets unavailable ({type(exc).__name__}); check notebook access.")
try:
    from google.colab import userdata
    secret_readers.append(userdata.get)
except ImportError:
    pass

required_key = "GOOGLE_API_KEY" if AI_PROVIDER == "gemini" else None
names = ["PEXELS_API_KEY", "HF_TOKEN"]
names.append("GOOGLE_API_KEY" if AI_PROVIDER == "gemini" else "VLLM_API_KEY")
for name in names:
    value = os.environ.get(name, "").strip()
    for read_secret in secret_readers:
        if value:
            break
        try:
            value = (read_secret(name) or "").strip()
        except Exception:
            pass  # Missing optional secrets must not block another provider.
    if value:
        os.environ[name] = value
    print(f"{name}: {'configured' if value else 'not configured'}")
del value

if required_key and not os.environ.get(required_key, "").strip():
    raise RuntimeError("Add GOOGLE_API_KEY to Colab/Kaggle Secrets or the environment for Gemini.")

if AI_PROVIDER == "vllm":
    from clipping.providers.config import vllm_config_values

    if START_LOCAL_VLLM:
        VLLM_BASE_URL = f"http://127.0.0.1:{VLLM_PORT}"
        if not os.environ.get("VLLM_API_KEY", "").strip():
            from secrets import token_urlsafe
            os.environ["VLLM_API_KEY"] = token_urlsafe(32)
    settings = vllm_config_values({
        "vllm_base_url": VLLM_BASE_URL, "vllm_model": VLLM_MODEL,
        "vllm_context_limit": VLLM_CONTEXT_LIMIT,
        "local_max_tokens": VLLM_MAX_TOKENS, "vllm_timeout": VLLM_TIMEOUT,
    })
    if START_LOCAL_VLLM and settings["vllm_context_limit"] is None:
        raise ValueError("Set VLLM_CONTEXT_LIMIT before starting a local server.")
    if settings["vllm_context_limit"] and settings["local_max_tokens"] >= settings["vllm_context_limit"]:
        raise ValueError("VLLM_MAX_TOKENS must be smaller than VLLM_CONTEXT_LIMIT.")
    for name, field in {
        "VLLM_BASE_URL": "vllm_base_url", "VLLM_MODEL": "vllm_model",
        "VLLM_CONTEXT_LIMIT": "vllm_context_limit", "VLLM_MAX_TOKENS": "local_max_tokens",
        "VLLM_TIMEOUT": "vllm_timeout",
    }.items():
        os.environ[name] = "" if settings[field] is None else str(settings[field])
    print("vLLM configuration ready. GOOGLE_API_KEY is not required.")
else:
    # Preserve the existing Gemini .env workflow; vLLM authentication stays in memory.
    env_text = "# Auto-generated by the notebook\n"
    for name in ("GOOGLE_API_KEY", "PEXELS_API_KEY", "HF_TOKEN"):
        env_text += f"{name}={os.environ.get(name, '')}\n"
    Path(".env").write_text(env_text, encoding="utf-8")
    print("Gemini .env created. Secret values are not displayed.")


### Optional local vLLM setup

With `START_LOCAL_VLLM = False`, the next cells skip installation and startup. With `True`, run Prepare once, then Start/Eval. The existing Kaggle helpers install application/server dependencies outside the notebook kernel. If setup or evaluation fails, fix the error before clipping. Report: `outputs/vllm-eval-short.json`; server log: `logs/kaggle-vllm.log`.

For another video using the same running server, rerun only the cookie/pipeline cells. Use Stop when finished; stop the server before repeating Prepare.


In [ ]:
# Prepare isolated application and vLLM environments (opt-in)
LOCAL_VLLM_PREPARED = False
LOCAL_VLLM_READY = False
if AI_PROVIDER == "vllm" and START_LOCAL_VLLM:
    from scripts.kaggle_python import bootstrap_environment, ensure_python
    from scripts.kaggle_storage import check_storage, prepare_storage
    from scripts.kaggle_studio import runtime_environment

    # Reject unsupported GPUs before downloading Python or installing packages.
    subprocess.run(
        [sys.executable, "scripts/kaggle_vllm.py", "preflight", "--gpu-index", str(GPU_INDEX)],
        cwd=PROJECT_DIR, check=True,
    )
    workspace = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else PROJECT_DIR
    storage = prepare_storage(PROJECT_DIR, workspace, mode="temporary")
    check_storage(storage, 1.0)
    cache_env = storage.cache_environment()
    BOOTSTRAP_PYTHON = ensure_python(storage.python_cache)
    setup_env = bootstrap_environment(storage.python_cache) | cache_env
    APP_ENV = storage.root / "kaggle-studio-cpu" / "venv"
    APP_PYTHON = str(APP_ENV / "bin/python")
    VLLM_ENV = storage.vllm_env
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_studio.py", "setup",
         "--venv", str(APP_ENV), "--torch-profile", "cpu"],
        cwd=PROJECT_DIR, env=setup_env, check=True,
    )
    check_storage(storage, 1.0)
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "setup", "--venv", str(VLLM_ENV)],
        cwd=PROJECT_DIR, env=setup_env, check=True,
    )
    check_storage(storage, 1.0)
    LOCAL_VLLM_PREPARED = True
    print("Application and vLLM environments ready; the notebook kernel is unchanged.")
else:
    print("Local vLLM installation skipped.")


In [ ]:
# Start local vLLM and evaluate one short transcript (opt-in)
LOCAL_VLLM_READY = False
if AI_PROVIDER == "vllm" and START_LOCAL_VLLM:
    if not LOCAL_VLLM_PREPARED:
        raise RuntimeError("Run the isolated vLLM preparation cell successfully first.")
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "start",
         "--venv", str(VLLM_ENV), "--gpu-index", str(GPU_INDEX),
         "--model", os.environ["VLLM_MODEL"], "--port", str(VLLM_PORT),
         "--context", os.environ["VLLM_CONTEXT_LIMIT"], "--gpu-memory-utilization", "0.85"],
        cwd=PROJECT_DIR, env=bootstrap_environment(storage.python_cache) | cache_env, check=True,
    )
    EVAL_OUTPUT = PROJECT_DIR / "outputs" / "vllm-eval-short.json"
    subprocess.run(
        [APP_PYTHON, "scripts/eval_local_llm.py",
         "--transcript", "tests/fixtures/eval_transcript_short.txt",
         "--clips", "1", "--output", str(EVAL_OUTPUT)],
        cwd=PROJECT_DIR, env=runtime_environment(APP_ENV) | cache_env, check=True,
    )
    LOCAL_VLLM_READY = True
    print(f"vLLM evaluation passed. Report: {EVAL_OUTPUT}")
else:
    print("Local start skipped. For vLLM, the configured external server must already be running.")


## 4. Run the Pipeline 🎬

This cell runs the full AI clipping pipeline:

1. **Download** — Downloads the YouTube video
2. **Transcribe** — Converts speech to text using Faster-Whisper
3. **Analyze** — Your selected Gemini or vLLM provider picks the most engaging moments
4. **Render** — Generates clips with face-tracking, subtitles, and hook teasers

> ⏱️ **Estimated time**: 5–15 minutes depending on video length and number of clips.

> ⚠️ Make sure you have set the `YOUTUBE_URL` in the Configuration section above!

### YouTube cookies (optional)

If YouTube asks you to sign in or blocks an anonymous download, use a **Netscape-format cookies.txt** export from your browser. Run the project setup first, then upload the file and set `YOUTUBE_COOKIES_FILE` in the next cell:

- **Colab:** upload through the Files sidebar, then use `/content/cookies.txt`.
- **Kaggle:** attach a private Input, then copy its file path (for example `/kaggle/input/my-private-cookies/cookies.txt`).
- **Other locations:** use an absolute path, or a path relative to the notebook's working directory. Paths containing spaces are supported.

The default reads `OSC_YOUTUBE_COOKIES_FILE` from the environment. Replace it with a path string to choose a file manually. Leave it empty to keep auto-detecting `youtube_cookies.txt` in the existing locations; without a detected file, the download runs without cookies. An explicit path that is missing or points to a directory stops the setup with an error.

Run the cookie setup cell again after changing the path. Keep cookie files private and out of shared notebook outputs. Cookies may help with authentication, but do not guarantee a successful download.


In [ ]:
# YouTube Cookies Setup (Optional)
import os
import shlex
import shutil

# Replace the value with your uploaded file path, or leave empty for auto-detection.
# Example: YOUTUBE_COOKIES_FILE = "/content/cookies.txt"
YOUTUBE_COOKIES_FILE = os.environ.get("OSC_YOUTUBE_COOKIES_FILE", "")

YT_COOKIES_ARG = ""
COOKIES_PATH = os.path.abspath("youtube_cookies.txt")
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

if YOUTUBE_COOKIES_FILE.strip():
    source_cookie = os.path.abspath(os.path.expanduser(YOUTUBE_COOKIES_FILE.strip()))
    if not os.path.isfile(source_cookie):
        raise FileNotFoundError(
            "YOUTUBE_COOKIES_FILE must point to an existing cookies.txt file. "
            "Upload it to Colab or attach a private Kaggle Input, then copy its file path."
        )
else:
    source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        # Copy contents only: Kaggle Input files may be read-only.
        shutil.copyfile(source_cookie, COOKIES_PATH)
    print(f"YouTube cookies active: {COOKIES_PATH}")
    YT_COOKIES_ARG = f"--yt-cookies {shlex.quote(COOKIES_PATH)}"
else:
    print("youtube_cookies.txt not found. The download will run without cookies.")


In [ ]:
# Run the clipping pipeline with the selected provider
import shlex

if AI_PROVIDER == "vllm" and START_LOCAL_VLLM and not LOCAL_VLLM_READY:
    raise RuntimeError("Run the local vLLM Start/Eval cell successfully before clipping.")

run_whisper_device = WHISPER_DEVICE
run_whisper_compute = WHISPER_COMPUTE_TYPE
if AI_PROVIDER == "vllm" and START_LOCAL_VLLM:
    run_whisper_device, run_whisper_compute = "cpu", "int8"
    print("Local vLLM uses the GPU; Whisper will use CPU/int8.")

command = [
    APP_PYTHON, "main.py",
    "--url", YOUTUBE_URL, "--clips", str(TOTAL_CLIPS), "--ratio", ASPECT_RATIO,
    "--font-style", FONT_STYLE, "--hook-duration", str(HOOK_DURATION),
    "--words-per-sub", str(WORDS_PER_SUBTITLE), "--ai-provider", AI_PROVIDER,
    "--whisper-model", WHISPER_MODEL, "--whisper-device", run_whisper_device,
    "--whisper-compute-type", run_whisper_compute,
]
if AI_PROVIDER == "vllm":
    # Process long transcripts in windows instead of requiring one giant prompt.
    command.append("--vllm-chunk-transcript")
command.extend(shlex.split(YT_COOKIES_ARG))
pipeline_env = (runtime_environment(APP_ENV) | cache_env
                if APP_ENV is not None else os.environ.copy())
subprocess.run(command, cwd=PROJECT_DIR, env=pipeline_env, check=True)


### Stop local vLLM when finished

Keep `STOP_LOCAL_VLLM = False` while processing videos. Set it to `True` and run this cell when finished; external servers are not stopped.


In [ ]:
# Stop the managed local server only when you are finished
STOP_LOCAL_VLLM = False  # Set True and run this cell after all clipping jobs are complete.
if STOP_LOCAL_VLLM and START_LOCAL_VLLM:
    subprocess.run(
        [sys.executable, "scripts/kaggle_vllm.py", "stop"], cwd=PROJECT_DIR, check=True,
    )
    LOCAL_VLLM_READY = False
    print("Managed local vLLM stopped. Re-run Start/Eval before another vLLM job.")


## 5. View & Download Results

After the pipeline completes, your clips are saved in the `outputs/` folder.
Run the cells below to preview and download them.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# List all generated output files
# ──────────────────────────────────────────────────────────────────────
import os

outputs_dir = "outputs"
if os.path.exists(outputs_dir):
    files = os.listdir(outputs_dir)
    video_files = [f for f in files if f.endswith(".mp4")]
    print(f"🎬 Generated {len(video_files)} clip(s):\n")
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(outputs_dir, f)) / (1024 * 1024)
        icon = "🎥" if f.endswith(".mp4") else "🖼️" if f.endswith((".jpg", ".png")) else "📄"
        print(f"   {icon} {f} ({size_mb:.1f} MB)")
else:
    print("❌ No outputs found. Make sure the pipeline ran successfully.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Download all output files as a ZIP archive
# ──────────────────────────────────────────────────────────────────────
import shutil
from google.colab import files

if os.path.exists(outputs_dir) and os.listdir(outputs_dir):
    zip_name = "clipping_outputs"
    shutil.make_archive(zip_name, "zip", outputs_dir)
    print(f"📦 Created {zip_name}.zip — downloading now...")
    files.download(f"{zip_name}.zip")
else:
    print("❌ No outputs to download.")

---

## 📖 Troubleshooting

| Problem | Solution |
|---------|----------|
| `GOOGLE_API_KEY not found` | Required for Gemini; add it to Secrets or select vLLM |
| `CUDA out of memory` | Use a smaller Whisper model (`small` or `medium`) |
| `float16 not supported` | Change `WHISPER_COMPUTE_TYPE` to `float32` |
| Pipeline takes too long | Reduce `TOTAL_CLIPS` to `1` for testing |
| No B-roll footage | Add `PEXELS_API_KEY` to Colab Secrets |

## 🔗 Links

- **GitHub**: [NaufalRizqullah/opensource-clipping](https://github.com/NaufalRizqullah/opensource-clipping)
- **Full Documentation**: See the [README](https://github.com/NaufalRizqullah/opensource-clipping/blob/main/README.md) for all CLI options
- **Get Gemini API Key**: [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
- **Get Pexels API Key**: [pexels.com/api](https://www.pexels.com/api/)